# Lab 03: Scikit-learn Regression Pipelines

In this lab, you will use scikit-learn pipelines to predict ocean-water temperature from oceanographic measurements. Complete every code and written-response space, then run the notebook from top to bottom before submitting.

## Learning outcomes

By the end of this lab, you should be able to:

- split regression data reproducibly;
- build, evaluate, and interpret preprocessing-and-modeling pipelines;
- tune pipeline hyperparameters with exhaustive and randomized search;
- preprocess mixed numeric and categorical data with a `ColumnTransformer`; and
- serialize a fitted model that accepts new raw observations.

## Data and submission

`data/ocean_data.csv` includes the target, `T_degC`. `data/ocean_data_newvalues.csv` has the same predictor columns but no target. Do not alter either data file. Submit this completed notebook, your generated `uv.lock`, and `lab-03-final-model.joblib` as described in the README.

In [ ]:
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

RANDOM_STATE = 307
TARGET = "T_degC"
CATEGORICAL_FEATURES = ["Wea", "Cloud_Typ", "Cloud_Amt", "Visibility"]
DATA_DIRECTORY = Path("data")
TRAINING_DATA_PATH = DATA_DIRECTORY / "ocean_data.csv"
NEW_DATA_PATH = DATA_DIRECTORY / "ocean_data_newvalues.csv"


In [ ]:
data = pd.read_csv(TRAINING_DATA_PATH)
new_data = pd.read_csv(NEW_DATA_PATH)

assert TARGET in data.columns
assert TARGET not in new_data.columns
assert set(new_data.columns) == set(data.columns) - {TARGET}

display(data.head())
print(f"Training rows: {len(data):,}")
print(f"New-value rows: {len(new_data):,}")

## 1. Prepare the data

For Sections 1–5, use only numeric predictors: exclude `Wea`, `Cloud_Typ`, `Cloud_Amt`, and `Visibility`. Use `T_degC` as the response. Create a 75%/25% train/test split with `random_state=307`. Keep the split variables available for later sections.

In [ ]:
# YOUR CODE HERE
# Define the numeric predictor columns, X_numeric, y, Xtrain, Xtest, ytrain, and ytest.

## 2. Baseline numeric pipeline

Build and fit a `Pipeline` for the numeric training data with these steps, in order:

1. `SimpleImputer(strategy="mean")`
2. `PolynomialFeatures(degree=2, include_bias=False)`
3. `StandardScaler()`
4. `LinearRegression()`

Report the training MSE and test MSE. Then compare the test MSE with the variance of `ytest` and interpret what that comparison says about predictability.

In [ ]:
# YOUR CODE HERE
# Build and fit the baseline numeric pipeline.

In [ ]:
# YOUR CODE HERE
# Compute and clearly print the training MSE, test MSE, and variance of ytest.

### Response — Section 2

Compare the test MSE to the variance of `ytest`. What does this say about the predictability of water temperature with this model?

_Write your response here._

## 3. Coefficient interpretation

Use the fitted baseline pipeline to obtain the polynomial-feature names and linear-regression coefficients. Order the features by coefficient value and report:

- the three features with the largest positive coefficients;
- the three features with the largest negative coefficients; and
- the fitted intercept.

Use a clearly labeled table or printed result so a grader can find all seven requested values.

In [ ]:
# YOUR CODE HERE
# Extract feature names and coefficients, then display the requested positive and negative features and intercept.

## 4. Exhaustive hyperparameter tuning

Build a numeric K-nearest-neighbors regression pipeline with imputation, polynomial features, standardization, and `KNeighborsRegressor`. Use `GridSearchCV` with 10-fold cross-validation, `scoring="neg_mean_squared_error"`, and these candidates:

- imputation strategy: `"mean"` or `"median"`;
- polynomial degree: 1, 2, or 3;
- number of neighbors: 5 through 100 by 5; and
- weights: `"uniform"` or `"distance"`.

Time the fit with `perf_counter`. Report the best hyperparameters, the best cross-validated MSE (with the sign corrected), elapsed time, and test MSE. Then compare the tuned KNN pipeline with the Section 2 baseline.

In [ ]:
# YOUR CODE HERE
# Build the numeric KNN pipeline and parameter grid, then fit GridSearchCV.

In [ ]:
# YOUR CODE HERE
# Print best parameters, positive MSE, elapsed time, and test MSE for the best estimator.

### Response — Section 4

Is the optimized KNN pipeline better at predicting water temperature than the Section 2 baseline? Support your answer with the test MSE values.

_Write your response here._

## 5. Randomized hyperparameter tuning

Repeat the Section 4 tuning task with `RandomizedSearchCV`. Use the same candidate values, 10-fold cross-validation, `scoring="neg_mean_squared_error"`, `n_iter=20`, and `random_state=307`. Report the best hyperparameters, best cross-validated MSE, elapsed time, and test MSE.

Compare the randomized-search model with the Section 2 baseline, then compare randomized and exhaustive search in terms of quality of the selected model and computation time.

In [ ]:
# YOUR CODE HERE
# Build and fit RandomizedSearchCV using the Section 4 candidate values.

In [ ]:
# YOUR CODE HERE
# Print best parameters, positive MSE, elapsed time, and test MSE for the best estimator.

### Response — Section 5(a)

Is the randomized-search pipeline better at predicting water temperature than the Section 2 baseline? Support your answer with the test MSE values.

_Write your response here._

### Response — Section 5(b)

Compare exhaustive and randomized search. Discuss both the selected models and the elapsed times.

_Write your response here._

## 6. Mixed-feature pipeline

Now use every predictor, including `Wea`, `Cloud_Typ`, `Cloud_Amt`, and `Visibility`. Build a single pipeline that uses a `ColumnTransformer` to combine:

- a numeric pipeline with mean imputation, second-degree polynomial features without a bias term, and standardization; and
- a categorical pipeline with most-frequent imputation, `OneHotEncoder(sparse_output=False, handle_unknown="ignore")`, and `SelectPercentile(f_regression, percentile=50)`.

Fit a `KNeighborsRegressor(n_neighbors=20, weights="distance")`. Report training and test MSE, then explain whether the additional predictors improve predictability.

In [ ]:
# YOUR CODE HERE
# Define the numeric and categorical feature sets, build the ColumnTransformer pipeline, and fit it.

In [ ]:
# YOUR CODE HERE
# Compute and clearly print training and test MSE for the mixed-feature pipeline.

### Response — Section 6

Do the additional categorical predictors improve predictability? Support your answer with the relevant test MSE values.

_Write your response here._

## 7. More pipeline functionality

Skim the [scikit-learn pipeline documentation](https://scikit-learn.org/stable/modules/compose.html) and one relevant example. Identify one pipeline capability not used above and explain how it could help in a future analysis.

### Response — Section 7

_Write your response here._

## 8. Select and share a final model

Choose the fitted model that you expect to make the best predictions for new observations. Your chosen `final_model` must accept the raw columns in `new_data`; it must not require manual preprocessing before calling `.predict(new_data)`. You may refine a pipeline from this notebook if you document the change.

Save the fitted model as `lab-03-final-model.joblib`, reload it, and verify that the reloaded model can predict the rows in `new_data`. Do not commit generated predictions unless you choose to include them as supporting evidence.

In [ ]:
# YOUR CODE HERE
# Select a fitted final_model that accepts new_data without manual preprocessing.

In [ ]:
# YOUR CODE HERE
# Save final_model to lab-03-final-model.joblib, reload it, and verify predictions on new_data.

## Final check

Before committing, restart the kernel and run all cells in order. Confirm that your notebook contains your code, printed results, and written responses; that `lab-03-final-model.joblib` exists; and that `uv.lock` was generated by `uv sync`.